In [ ]:
import numpy as np
import pandas as pd

import astropy.units as u

from astropy.cosmology import Planck18
from scipy.integrate import quad

In [ ]:
# Power-law fits from plot_Av.ipynb

# All foregrounds
popt = np.array([0.12306992, -1.79492352])
pcov = np.array([[0.00127918, -0.00273679], [-0.00273679, 0.19112034]])
fg = pd.read_parquet("../results/samples/dp1_pai24/foreground.parquet")

# Red sample
# popt = np.array([0.19435621, -1.92688228])
# pcov = np.array([[0.00362424, -0.00458696], [-0.00458696,  0.27703364]])
# fg = pd.read_parquet("../results/samples/dp1_red/foreground.parquet")

# Blue sample
# popt = np.array([0.03138386, -2.3822535])
# pcov = np.array([[1.54475284e-03, 7.87818670e-02], [7.87818670e-02, 7.19020546e+00]])
# fg = pd.read_parquet("../results/samples/dp1_blue/foreground.parquet")

In [ ]:
def Av(r, norm, index):
    """A_V power law"""
    return norm * (r / 10) ** index


kv = 2e4 * u.cm**2 / u.g

prefactor = np.log(10) / 2.5 / kv
prefactor = prefactor.to_value(u.Msun / u.kpc**2)


def calc_dust_mass(norm, index):
    """Integrated dust mass (Msun) for the given Av profile parameters."""
    Sigma_dust = lambda r: prefactor * Av(r, norm, index)
    integrand = lambda r: 2 * np.pi * Sigma_dust(r) * r
    return quad(integrand, 20 / Planck18.h, 110 / Planck18.h)[0]


# Central estimate from the best-fit parameters
dust_mass = calc_dust_mass(*popt)

# Propagate the uncertainty in popt (via pcov) with a Monte Carlo draw.
# The mass is nonlinear in `index`, so we sample rather than linearize.
rng = np.random.default_rng(0)
samples = rng.multivariate_normal(popt, pcov, size=100_000)
dust_mass_samples = np.array([calc_dust_mass(n, i) for n, i in samples])

# Asymmetric 1-sigma error bars from the 16th/50th/84th percentiles
dm_lo, dm_med, dm_hi = np.percentile(dust_mass_samples, [16, 50, 84])
dust_mass_errlo = dm_med - dm_lo
dust_mass_errhi = dm_hi - dm_med

print(f"dust mass = {dm_med:.2e} (-{dust_mass_errlo:.2e} / +{dust_mass_errhi:.2e})")

In [ ]:
halo_mass = np.nanmedian(10**fg.halo_mass_log)
stellar_mass = np.nanmedian(10**fg.stellar_mass_log)


def summarize(samples):
    """Return (median, -err, +err) from the 16/50/84 percentiles."""
    lo, med, hi = np.percentile(samples, [16, 50, 84])
    return med, med - lo, hi - med


# Propagate the dust-mass uncertainty through each ratio
halo_ratio = summarize(dust_mass_samples / halo_mass)
stellar_ratio = summarize(dust_mass_samples / stellar_mass)

print(
    f"dust/halo    = {halo_ratio[0]:.2e} (-{halo_ratio[1]:.2e} / +{halo_ratio[2]:.2e})"
)
print(
    f"dust/stellar = {stellar_ratio[0]:.2e} (-{stellar_ratio[1]:.2e} / +{stellar_ratio[2]:.2e})"
)

In [ ]:
stellar_mass / 1e10

In [ ]:
halo_mass / 1e11